# Where do Claude Code's tokens go? Harness features compared

The same small coding task (a brick-breaker game in one HTML file) was run 5 times under each of 7 Claude Code settings on Opus 5.5: default, safe mode (installed plugins off), a minimal harness, a subagent, plan mode, a project skill and a PostToolUse hook.

This notebook reproduces the main numbers from the CSV and shows why **token counts and cost tell different stories** once prompt caching is involved.

Source, method and raw logs: https://tokensave.app/data/claude-code-harness-cost · DOI [10.5281/zenodo.23465633](https://doi.org/10.5281/zenodo.23465633) · CC BY 4.0

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/claude-code-harness-cost.csv', recursive=True) or ['claude-code-harness-cost.csv'])[0]
df = pd.read_csv(path)
df.head()

## Median per setting and change versus default

In [ ]:
order = ['default', 'safemode', 'minimal', 'hook', 'skill', 'plan', 'subagent']
cols = ['input_tokens', 'cache_read_tokens', 'output_tokens', 'api_cost_usd', 'seconds', 'file_chars']
med = df.groupby('condition')[cols].median().loc[order]
base = med.loc['default']
pct = ((med / base - 1) * 100).round(0).astype(int)
summary = med.round(3).join(pct[['input_tokens', 'api_cost_usd']].add_suffix('_vs_default_%'))
summary['games_working'] = df.groupby('condition')['works'].sum().loc[order].astype(str) + ' / 5'
summary

## Tokens vs cost

Plan mode, a skill and a hook all add model turns, and every turn re-reads the context. Most of that re-read comes from the prompt cache, which is billed at a fraction of the normal input price, so token counts rise much faster than cost. A subagent starts a fresh context of its own, so it raises both.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = range(len(order))
ax.bar([i - 0.2 for i in x], pct.loc[order, 'input_tokens'], width=0.4, label='input tokens vs default (%)')
ax.bar([i + 0.2 for i in x], pct.loc[order, 'api_cost_usd'], width=0.4, label='API cost vs default (%)')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(order, rotation=20)
ax.set_ylabel('% change vs default (median of 5 runs)')
ax.legend(); ax.set_title('Same task, 7 Claude Code settings')
plt.tight_layout(); plt.show()

## Share of input read from the cache

In [ ]:
share = (med['cache_read_tokens'] / med['input_tokens'] * 100).round(0).astype(int)
share.rename('cache_read_share_%').to_frame()

## Notes

- Safe mode and the minimal harness also produced shorter games (see `file_chars` and `output_tokens`), so part of their cost drop is output, not input.
- One small task and five runs per setting. On larger tasks a plan, a skill or a subagent may save rework that this task never needed.
- Cost is the API list-price estimate Claude Code reports; subscription plans are not billed per token.
- Related datasets (CLAUDE.md size, Claude Code vs Codex, web search, prompt language): https://tokensave.app/data/